# Backtesting

This notebook is a scaffold for single-asset strategy backtesting workflows in the Pricing folder. It loads shared Single Asset Profile defaults from params.py, builds a simple moving-average regime signal, adds an inverse-volatility sizing strategy, and compares both approaches to buy-and-hold.

In [ ]:
import warnings
from pathlib import Path
import runpy

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf

warnings.filterwarnings("ignore")

In [ ]:
PARAMS_FILE_CANDIDATES = []
for base_path in (Path.cwd(), *Path.cwd().parents):
    PARAMS_FILE_CANDIDATES.extend([
        base_path / "params.py",
        base_path / "Notebooks" / "Single Asset Profile" / "params.py",
    ])

seen_params_paths = set()
for params_file in PARAMS_FILE_CANDIDATES:
    params_file = params_file.resolve()
    if params_file in seen_params_paths:
        continue
    seen_params_paths.add(params_file)
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()
resolve_time_frame_map = notebook_params["resolve_time_frame_map"]

ticker_str = single_asset_params["ticker_str"]
interval = single_asset_params["interval"]
period = single_asset_params["period"]
trading_strategy = single_asset_params["trading_strategy"]
time_frame_map = resolve_time_frame_map(trading_strategy)

single_asset_params

In [ ]:
price_frame = yf.Ticker(ticker_str).history(period=period, interval=interval).copy()
if price_frame.empty:
    raise ValueError(f"No price history returned for {ticker_str}.")

price_frame.index = pd.to_datetime(price_frame.index).tz_localize(None)
price_frame = price_frame.sort_index()
price_frame["daily_return"] = price_frame["Close"].pct_change().fillna(0.0)
price_frame["short_ma"] = price_frame["Close"].rolling(time_frame_map["short"]).mean()
price_frame["long_ma"] = price_frame["Close"].rolling(time_frame_map["long"]).mean()
price_frame["signal"] = (price_frame["short_ma"] > price_frame["long_ma"]).astype(float)
price_frame["position"] = price_frame["signal"].shift(1).fillna(0.0)
price_frame["strategy_return"] = price_frame["position"] * price_frame["daily_return"]

price_frame["rolling_vol_21d"] = price_frame["daily_return"].rolling(21).std() * np.sqrt(252)
price_frame["inverse_vol_raw"] = 1 / price_frame["rolling_vol_21d"].replace(0, np.nan)
price_frame["inverse_vol_scale"] = price_frame["inverse_vol_raw"] / price_frame["inverse_vol_raw"].expanding().mean()
price_frame["inverse_vol_position"] = price_frame["inverse_vol_scale"].clip(lower=0.0, upper=1.5).shift(1).fillna(0.0)
price_frame["inverse_vol_return"] = price_frame["inverse_vol_position"] * price_frame["daily_return"]

price_frame["buy_hold_index"] = (1 + price_frame["daily_return"]).cumprod()
price_frame["strategy_index"] = (1 + price_frame["strategy_return"]).cumprod()
price_frame["inverse_vol_index"] = (1 + price_frame["inverse_vol_return"]).cumprod()

backtest_summary = pd.DataFrame([
    {
        "series": "Buy and hold",
        "cumulativeReturnPct": round((price_frame["buy_hold_index"].iloc[-1] - 1) * 100, 2),
        "annualizedVolPct": round(price_frame["daily_return"].std() * np.sqrt(252) * 100, 2),
    },
    {
        "series": "MA crossover strategy",
        "cumulativeReturnPct": round((price_frame["strategy_index"].iloc[-1] - 1) * 100, 2),
        "annualizedVolPct": round(price_frame["strategy_return"].std() * np.sqrt(252) * 100, 2),
    },
    {
        "series": "Inverse volatility strategy",
        "cumulativeReturnPct": round((price_frame["inverse_vol_index"].iloc[-1] - 1) * 100, 2),
        "annualizedVolPct": round(price_frame["inverse_vol_return"].std() * np.sqrt(252) * 100, 2),
    },
])

backtest_summary

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=price_frame.index,
        y=price_frame["buy_hold_index"],
        name="Buy and hold",
        line={"color": "#94a3b8", "width": 2},
    )
)
fig.add_trace(
    go.Scatter(
        x=price_frame.index,
        y=price_frame["strategy_index"],
        name="MA crossover strategy",
        line={"color": "#22c55e", "width": 2.5},
    )
)
fig.add_trace(
    go.Scatter(
        x=price_frame.index,
        y=price_frame["inverse_vol_index"],
        name="Inverse volatility strategy",
        line={"color": "#38bdf8", "width": 2.5},
    )
)
fig.update_layout(
    title=f"{ticker_str} backtest equity curves",
    template="plotly_dark",
    xaxis_title="Date",
    yaxis_title="Growth of $1",
    height=500,
)
fig.show(config={"responsive": True, "displaylogo": False})